# GridPulse — Week 3 Data Exploration

**Project:** P05 — GridPulse Campus Energy Command Center
**Team:** Team 05
**Week:** 3

Goal: Load the official GridPulse source data, inspect schemas and grain, profile data quality issues, answer an initial business question, and demonstrate Delta storage and lineage without performing production Bronze/Silver/Gold transformations.


In [ ]:
BUILDINGS_PATH = "/Volumes/workspace/default/gridpulse/buildings.json"
READINGS_PATH = "/Volumes/workspace/default/gridpulse/consumption_readings.parquet"
METERS_PATH = "/Volumes/workspace/default/gridpulse/meters.csv"
TARIFFS_PATH = "/Volumes/workspace/default/gridpulse/tariffs.csv"

print("===== GRIDPULSE SOURCE PATHS =====")
print("Buildings:", BUILDINGS_PATH)
print("Consumption readings:", READINGS_PATH)
print("Meters:", METERS_PATH)
print("Tariffs:", TARIFFS_PATH)


In [ ]:
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.compute as pc

buildings_df = spark.read.option("multiLine", True).json(BUILDINGS_PATH)
meters_df = spark.read.option("header", True).option("inferSchema", True).csv(METERS_PATH)
tariffs_df = spark.read.option("header", True).option("inferSchema", True).csv(TARIFFS_PATH)

# Official readings file uses TIMESTAMP(NANOS), so convert nanosecond timestamps
# to microseconds with PyArrow before creating the Spark DataFrame.
readings_table = pq.read_table(READINGS_PATH)

new_fields = []
new_columns = []

for field, column in zip(readings_table.schema, readings_table.columns):
    if pa.types.is_timestamp(field.type) and field.type.unit == "ns":
        field = field.with_type(pa.timestamp("us"))
        column = pc.cast(column, pa.timestamp("us"))

    new_fields.append(field)
    new_columns.append(column)

readings_table = pa.Table.from_arrays(
    new_columns,
    schema=pa.schema(new_fields)
)

readings_df = spark.createDataFrame(readings_table.to_pandas())

print("===== DATA LOAD COMPLETE =====")
print("Buildings:", buildings_df.count())
print("Meters:", meters_df.count())
print("Readings:", readings_df.count())
print("Tariffs:", tariffs_df.count())


In [ ]:
print("===== SAMPLE RECORDS =====")
display(buildings_df.limit(5))
display(meters_df.limit(5))
display(readings_df.limit(5))
display(tariffs_df.limit(5))


## Schema Inspection


In [ ]:
print("===== BUILDINGS SCHEMA =====")
buildings_df.printSchema()

print("===== METERS SCHEMA =====")
meters_df.printSchema()

print("===== READINGS SCHEMA =====")
readings_df.printSchema()

print("===== TARIFFS SCHEMA =====")
tariffs_df.printSchema()


## Grain, Counts, Distinct Keys, and Nulls


In [ ]:
from pyspark.sql import functions as F

print("===== ROW COUNTS =====")
print("Buildings:", buildings_df.count())
print("Meters:", meters_df.count())
print("Readings:", readings_df.count())
print("Tariffs:", tariffs_df.count())

print("\n===== DISTINCT KEYS =====")
print("Distinct building_id:", buildings_df.select("building_id").distinct().count())
print("Distinct meter_id:", meters_df.select("meter_id").distinct().count())
print("Distinct reading_id:", readings_df.select("reading_id").distinct().count())
print("Distinct tariff_id:", tariffs_df.select("tariff_id").distinct().count())

print("\n===== NULL COUNTS IN READINGS =====")
null_exprs = [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in readings_df.columns]
display(readings_df.select(null_exprs))


## Duplicate Key Checks


In [ ]:
print("===== DUPLICATE reading_id VALUES =====")
duplicate_reading_ids = (
    readings_df
    .groupBy("reading_id")
    .count()
    .filter(F.col("count") > 1)
)
display(duplicate_reading_ids)

print("===== DUPLICATE reading_id RECORDS =====")
display(
    readings_df.join(
        duplicate_reading_ids.select("reading_id"),
        "reading_id",
        "inner"
    ).orderBy("reading_id", "source_record_id")
)

print("===== DUPLICATE (meter_id, reading_ts) =====")
natural_key_duplicates = (
    readings_df
    .groupBy("meter_id", "reading_ts")
    .count()
    .filter(F.col("count") > 1)
)
display(natural_key_duplicates)


## Numeric Profiling and Suspicious Values


In [ ]:
numeric_columns = [
    "energy_kwh",
    "active_power_kw",
    "voltage_v",
    "current_a",
    "power_factor"
]

print("===== NUMERIC SUMMARY =====")
display(readings_df.select(numeric_columns).summary())

print("===== SUSPICIOUS VALUE COUNTS =====")
print("energy_kwh < 0:", readings_df.filter(F.col("energy_kwh") < 0).count())
print("active_power_kw < 0:", readings_df.filter(F.col("active_power_kw") < 0).count())
print("voltage_v <= 0:", readings_df.filter(F.col("voltage_v") <= 0).count())
print("current_a < 0:", readings_df.filter(F.col("current_a") < 0).count())
print("power_factor < 0:", readings_df.filter(F.col("power_factor") < 0).count())
print("power_factor > 1:", readings_df.filter(F.col("power_factor") > 1).count())


## Relationship and Referential Checks


In [ ]:
print("===== READINGS WITHOUT A METER REFERENCE =====")
orphan_readings = readings_df.join(
    meters_df.select("meter_id").distinct(),
    "meter_id",
    "left_anti"
)
print("Count:", orphan_readings.count())
display(orphan_readings)

print("===== METERS WITHOUT A BUILDING REFERENCE =====")
orphan_meters_building = meters_df.join(
    buildings_df.select("building_id").distinct(),
    "building_id",
    "left_anti"
)
print("Count:", orphan_meters_building.count())
display(orphan_meters_building)

print("===== METERS WITHOUT A TARIFF PLAN REFERENCE =====")
orphan_meters_tariff = meters_df.join(
    tariffs_df.select("tariff_plan_id").distinct(),
    "tariff_plan_id",
    "left_anti"
)
print("Count:", orphan_meters_tariff.count())
display(orphan_meters_tariff)


## Effective-Date and Tariff Checks


In [ ]:
print("===== METERS WITH effective_from > effective_to =====")
invalid_meter_periods = meters_df.filter(F.col("effective_from") > F.col("effective_to"))
print("Count:", invalid_meter_periods.count())
display(invalid_meter_periods)

print("===== TARIFFS WITH effective_from > effective_to =====")
invalid_tariff_periods = tariffs_df.filter(F.col("effective_from") > F.col("effective_to"))
print("Count:", invalid_tariff_periods.count())
display(invalid_tariff_periods)


## Timestamp Alignment and Coverage


In [ ]:
print("===== 15-MINUTE ALIGNMENT CHECK =====")
misaligned_readings = readings_df.filter(
    (F.minute("reading_ts") % 15 != 0) |
    (F.second("reading_ts") != 0)
)
print("Readings not aligned to 15-minute intervals:", misaligned_readings.count())
display(misaligned_readings)

print("===== TIMESTAMP RANGE =====")
display(
    readings_df.select(
        F.min("reading_ts").alias("minimum_reading_ts"),
        F.max("reading_ts").alias("maximum_reading_ts")
    )
)

print("===== LATEST READINGS =====")
display(readings_df.orderBy(F.col("reading_ts").desc()).limit(20))


## Missing-Interval Check


In [ ]:
expected_readings_per_meter = 2500

meter_reading_counts = (
    readings_df
    .groupBy("meter_id")
    .agg(
        F.count("reading_ts").alias("reading_count"),
        F.min("reading_ts").alias("min_reading_ts"),
        F.max("reading_ts").alias("max_reading_ts")
    )
    .withColumn(
        "expected_readings",
        F.lit(expected_readings_per_meter)
    )
    .withColumn(
        "missing_count",
        F.col("expected_readings") - F.col("reading_count")
    )
)

print("===== METERS WITH MISSING/EXTRA READING COUNTS =====")
display(
    meter_reading_counts
    .filter(F.col("reading_count") != F.col("expected_readings"))
    .orderBy("meter_id")
)


## Initial Business Question

Which meters have the highest total energy consumption in the observed source data?


In [ ]:
top_meters = (
    readings_df
    .groupBy("meter_id")
    .agg(
        F.round(F.sum("energy_kwh"), 2).alias("total_energy_kwh"),
        F.round(F.avg("active_power_kw"), 2).alias("avg_active_power_kw"),
        F.count("reading_id").alias("reading_count")
    )
    .orderBy(F.col("total_energy_kwh").desc())
    .limit(10)
)

display(top_meters)


## Learning-Only Delta Demo

This is a Week 3 learning/demo table only. It is not the official production Bronze layer.


In [ ]:
bronze_demo_table = "workspace.default.gridpulse_week03_bronze_demo_readings"

(
    readings_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(bronze_demo_table)
)

print("===== WEEK 3 DELTA DEMO =====")
print("Bronze demo table created:", bronze_demo_table)
print("Source readings:", readings_df.count())
print("Bronze demo rows:", spark.table(bronze_demo_table).count())


In [ ]:
print("===== DELTA TABLE DETAIL =====")
display(spark.sql(f"DESCRIBE DETAIL {bronze_demo_table}"))

print("===== DELTA TABLE HISTORY =====")
display(spark.sql(f"DESCRIBE HISTORY {bronze_demo_table}"))


## Learning-Only Lineage Demo


In [ ]:
lineage_view = "workspace.default.gridpulse_week03_lineage_demo_view"

spark.sql(f"""
CREATE OR REPLACE VIEW {lineage_view} AS
SELECT
    reading_id,
    meter_id,
    reading_ts,
    energy_kwh,
    active_power_kw,
    voltage_v,
    current_a,
    power_factor,
    reading_quality_flag
FROM {bronze_demo_table}
""")

print("===== WEEK 3 LINEAGE DEMO VIEW =====")
print("Lineage demo view created:", lineage_view)


## Week 3 Findings — Do Not Fix Yet

The source exploration identified data-quality findings that will be handled in later pipeline stages rather than silently changed during Week 3 exploration.

- 1 duplicate `reading_id`
- 1 duplicate `(meter_id, reading_ts)` natural key
- 1 negative `energy_kwh` value
- 1 reading with a missing meter reference (`MTR9999`)
- 1 meter with `effective_from > effective_to` (`MTR0120`)
- 1 reading not aligned to a 15-minute interval
- Reading-count/timestamp coverage anomalies were observed for some meters

These records are retained for later validation/quarantine logic. Week 3 is exploration only.


## Evidence to Capture

- `week03_01_source_files.png` — source files / Databricks volume
- `week03_02_dataframes.png` — loaded DataFrames
- `week03_03_schemas.png` — source schemas
- `week03_04_grain_counts_values.png` — counts, keys, duplicates, numeric profile
- `week03_05_relationship_checks.png` — referential/effective-date/timestamp checks
- `week03_06_bronze_demo.png` — learning-only Delta demo
- `week03_07_delta_history.png` — Delta history
- `week03_08_lineage_graph.png` — Catalog Explorer lineage graph

Also update `weekly_logs/week03_log.md`, `docs/data_dictionary.md`, `docs/requirements.md`, and `docs/pipeline_walkthrough.md`.
